In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Lab 07: EM Algorithm

In this session, we explore the frequentist parameter inference of the Gaussian Mixture Model (GMM). The GMM is defined as the following generative process: For each point $n$:
* Choose a cluster $z_n \in \lbrace 1, \dotsc, K \rbrace$ with probability $p(x_n = k) = \pi_k$
* Draw $x_n$ from normal distribution $\mathcal{N}(\mu_{z_n}, \Sigma_{z_n})$. 

The likelihood of the parameters $\theta$ of the GMM for the point $x_n$ is then:
$$
p_{\theta}(x_n) = \sum_{k=1}^K \pi_k \, \mathcal{N}(x_n: \mu_{k}, \Sigma_{k})
$$


In this session, we will compute the Maximum Likelihood estimator of $\theta$ using the EM algorithm. 


## A test dataset

To test our algorithm, we use the blob dataset generation of the scikit-learn library. This generator creates a dataset evenly distributed among $K$ multivariate normal distributions. We propose several variants of this dataset:
* _Standard:_ The points are evenly distributed among clusters, and all distributions have the same spherical covariance $\Sigma_k = \sigma^2 I$
* _Anisotropic:_ Differs from the standard case on the fact that the covariances are still the same, but not spherical.
* _Unequal variance:_ All distributions are different but spherical covariance. 
* _Unevenly distributied:_ All clusters have the same spherical covariance, but the points are unevenly distributed between clusters. 


In [ ]:
# Dataset hyperparameters

n_samples = 1500
seed= 3456789
transformation = [[0.60834549, -0.63667341], [-0.40887718, 0.85253229]]

In [ ]:
# Generation of the datasets

X, y = make_blobs(n_samples=n_samples, random_state=seed)
X_aniso = np.dot(X, transformation)  # Anisotropic blobs
X_varied, y_varied = make_blobs(
    n_samples=n_samples, cluster_std=[1.0, 2.5, 0.5], random_state=seed
)  # Unequal variance
X_filtered = np.vstack(
    (X[y == 0][:500], X[y == 1][:100], X[y == 2][:10])
)  # Unevenly sized blobs
y_filtered = [0] * 500 + [1] * 100 + [2] * 10

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 12))

axs[0, 0].scatter(X[:, 0], X[:, 1], c=y)
axs[0, 0].set_title("Mixture of Gaussian Blobs")

axs[0, 1].scatter(X_aniso[:, 0], X_aniso[:, 1], c=y)
axs[0, 1].set_title("Anisotropically Distributed Blobs")

axs[1, 0].scatter(X_varied[:, 0], X_varied[:, 1], c=y_varied)
axs[1, 0].set_title("Unequal Variance")

axs[1, 1].scatter(X_filtered[:, 0], X_filtered[:, 1], c=y_filtered)
axs[1, 1].set_title("Unevenly Sized Blobs")

plt.suptitle("Ground truth clusters").set_y(0.95)
plt.show()

## E Step

In the E Step, the goal is to compute the quantity:
$$
Q_n(\theta; \theta^{(t)}) = \mathbb{E}_{z_n \sim p(. | x_n)}[\log p_{\theta}(x_n, z_n) ]
$$
The total ELBO is the sum over the $Q_n$: 
$$
Q(\theta; \theta^{(t)}) = \sum_{n=1}^N Q_n(\theta; \theta^{(t)})
$$

#### Question 1

a) Assuming that we are fixing the parameter $\theta^{(t)}$, compute the posterior $r_{nk} = p(z_n = k | x_n)$.<br>
b) Compute $Q(\theta; \theta^{(t)})$. 


## M Step

In the M step, we maximize the $Q$ function over $\theta$. We will not do the whole calculation in this session, but we will just see how to estimate $(\pi_1, \dotsc, \pi_K)$. 

#### Question 2

Compute the derivate of $Q$ with respect to $\pi_k$. Show that the gradient does not vanish. This is incoherent: find the mistake and correct it!

The formulae of the optimized parameters are given in the lecture:

$$
\pi_k^{(t+1)} = \frac{1}{N} \sum_{n=1}^N r_{nk}^{(t)} = \frac{r_k}{N} \qquad \qquad \mu_k^{(t+1)} = \frac{\sum_{n=1}^N r_{nk}^{(t)} x_n}{r_k} \qquad \qquad \Sigma_k^{(t+1)} = \frac{1}{r_k}  \sum_{n=1}^N r_{nk}^{(t)} (x_n - \mu_k) (x_n - \mu_k)^T 
$$

In our code, we will group all the parameters in a dictionary:

In [ ]:
K = 3
d = 2

theta_t = { 'pi': np.ones((K,)) / K,
            'mu': np.random.rand(K, d), 
            'sigma': [np.eye(d) for _ in range(K)]}

In [ ]:
theta_t

#### Question 3

Implement the parameter update.

In [ ]:
def parameter_update(theta_t, X):
    ### Your code here
    return theta_t

In [ ]:
### Code used to compute the log-likelihood

def multivariate_normal_pdf_batch(xs, mu, sigma):
    """
    Compute the likelihood (PDF) of a multivariate normal distribution for multiple observations.
    
    Parameters:
    - xs: Observations (2D numpy array of shape (n_samples, n_features))
    - mu: Mean vector (1D numpy array of shape (n_features,))
    - sigma: Covariance matrix (2D numpy array of shape (n_features, n_features))
    
    Returns:
    - pdfs: 1D numpy array containing the PDF for each observation
    """
    n_samples, n_features = xs.shape
    sigma_det = np.linalg.det(sigma)  # Determinant of the covariance matrix
    sigma_inv = np.linalg.inv(sigma)  # Inverse of the covariance matrix

    # Normalization constant
    norm_constant = 1 / (np.sqrt((2 * np.pi) ** n_features * sigma_det))
    
    # Compute the exponent for each observation
    diffs = xs - mu  # Subtract mean from each observation
    exponents = -0.5 * np.einsum('ij,jk,ik->i', diffs, sigma_inv, diffs)  # Batch computation
    
    # Compute the PDF for each observation
    pdfs = norm_constant * np.exp(exponents)
    
    return pdfs

# Example usage
xs = np.array([
    [1.0, 2.0],
    [0.0, 1.0],
    [2.0, 1.5]
])  # Batch of observations (3 samples, 2 features)

mu = np.array([0.0, 0.0])  # Mean vector
sigma = np.array([[1.0, 0.5], [0.5, 1.0]])  # Covariance matrix

pdf_values = multivariate_normal_pdf_batch(xs, mu, sigma)
print("Likelihood (PDF) values for the observations:")
print(pdf_values)


def multivariate_normal_pdf_batch_cluster(xs, mus, sigmas, K):
    return np.array([multivariate_normal_pdf_batch(X, mus[k], sigmas[k]) for k in range(K)])


def loglikelihood_gmm(theta, X):
    likelihoods = theta['pi'][k] * multivariate_normal_pdf_batch_cluster(X, theta['mu'], theta['sigma'], K)
    log_n = np.log(likelihoods.sum(0))
    return log_n.sum()


#### Question 4

Implement the EM algorithm, taking as input the dataset ```X```, the initial ```theta``` as well as all the parameters you think are useful. You will need to define a termination criterion for your algorithm. What would be a debugging test that you might want to run?

In [ ]:
def EM(X, theta, ...):
    # Your code here
    pass

## GMM for clustering

We now propose to see how the GMM can be used for clustering. By design of the model, GMM is a _soft clustering_ method, in the sense that it provides the probability of a point to belong to a cluster, but not a cluster allocation. To make it a _hard clustering_, we allocate each point to its most likely cluster. 

In [ ]:
def allocate_points(X, theta, K):
    likelihoods = multivariate_normal_pdf_batch_cluster(X, theta['mu'], theta['sigma'], K)
    return np.argmax(likelihoods, axis=0)

The following code implements a visualization method for a produced clustering:

In [ ]:
def visualize_clustering(X, clusters):
    plt.scatter(X[:, 0], X[:, 1], c=clusters)
    plt.show()

In [ ]:
visualize_clustering(X, allocate_points(X, theta_t, K))

#### Question 5

Visualize the evolution of the clusters at each step of the EM for the four datasets. Test different initializations. Which one seems the most reasonable? How would you propose an initialization for $\theta$? Test the behaviour of the algorithm for $K > 3$. How would you choose the optimal number of clusters?